# CSC 729 Assignment 5
**Ntini, Sunny**

In [ ]:
!mamba install pandas
import numpy as np
import pandas as pd
from fractions import Fraction

pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)

---
# Problem 1: Classification

## a. Dataset description

- **Features:** Age: numerical, Income ($1000): numerical, Marital Status: categorical (Single / Married)
- **Label:** Buy (Yes / No), categorical binary class

In [ ]:
data = pd.DataFrame({
    "Customer": range(1, 13),
    "Age":      [25, 45, 30, 35, 50, 28, 40, 32, 38, 22, 29, 48],
    "Income":   [50, 100, 60, 40, 90, 70, 120, 55, 80, 65, 55, 95],
    "Marital":  ["Single", "Married", "Single", "Married", "Married", "Single",
                 "Married", "Single", "Married", "Single", "Married", "Single"],
    "Buy":      ["Yes", "No", "Yes", "Yes", "No", "Yes", "No", "Yes", "No", "Yes", "No", "Yes"],
}).set_index("Customer")

display(data)

Out of $N=12$ customers, 7 have Buy = Yes ($p_{\text{Yes}}=\frac{7}{12}$) and 5 have Buy = No ($p_{\text{No}}=\frac{5}{12}$).

$$\text{Gini}(D)=1-\sum_{i=1}^{k}{p_{i}^{2}}=1-\left[\left(\frac{7}{12}\right)^{2}+\left(\frac{5}{12}\right)^{2}\right]$$

$$\text{Gini}(D)=1-\left(\frac{49}{144}+\frac{25}{144}\right)=1-\frac{74}{144}=\frac{70}{144}\approx 0.4861$$

## b. Split on Income ($\text{income}\le 75$ vs. $\text{income}>75$)

- Income ≤ 75: 6 Yes, 1 No (7 customers)

$$\text{Gini}(L)=1-\left(\frac{6}{7}\right)^{2}-\left(\frac{1}{7}\right)^{2}=\frac{12}{49}\approx 0.2449$$

- Income > 75: 1 Yes, 4 No (5 customers)

$$\text{Gini}(G)=1-\left(\frac{1}{5}\right)^{2}-\left(\frac{4}{5}\right)^{2}=\frac{8}{25}=0.32$$

The weighted Gini after the split is:

$$\frac{7}{12}\left(\frac{12}{49}\right)+\frac{5}{12}\left(\frac{8}{25}\right)=\frac{29}{105}\approx 0.2762$$

## c. Split on Marital Status

- Single: 6 customers: 6 Yes, 0 No.  $\text{Gini}_{\text{Single}}=1-\left[\left(\frac{6}{6}\right)^{2}+\left(\frac{0}{6}\right)^{2}\right]=0$
- Married: 6 customers: 1 Yes, 5 No.  $\text{Gini}_{\text{Married}}=1-\left[\left(\frac{1}{6}\right)^{2}+\left(\frac{5}{6}\right)^{2}\right]=1-\frac{26}{36}=\frac{10}{36}\approx 0.2778$

The weighted Gini after the split is:

$$\frac{6}{12}(0)+\frac{6}{12}\left(\frac{5}{18}\right)=\frac{5}{36}\approx 0.1389$$

## d. Which split is better?

Split on **Marital Status**. Its weighted Gini (0.1389) is lower than Income's (0.2762), meaning it creates purer groups. Both reduce impurity from the original 0.4861, but the marital-status split reduces it more. It also produces one completely pure node, since all Single customers buy.

## e. Split on Age

Evaluating all 11 midpoint thresholds for $\text{Age}\le t$ vs. $\text{Age}>t$ across the $N=12$ customers sorted from lowest to highest.

| Threshold (t) | Left (Age <= t) | Left Gini | Right (Age > t) | Right Gini | Weighted Gini |
| :---: | :--- | :---: | :--- | :---: | :---: |
| 23.5 | n=1 (1 Yes, 0 No) | 0.0000 | n=11 (6 Yes, 5 No) | 0.4959 | 0.4545 |
| 26.5 | n=2 (2 Yes, 0 No) | 0.0000 | n=10 (5 Yes, 5 No) | 0.5000 | 0.4167 |
| 28.5 | n=3 (3 Yes, 0 No) | 0.0000 | n=9 (4 Yes, 5 No) | 0.4938 | 0.3704 |
| 29.5 | n=4 (3 Yes, 1 No) | 0.3750 | n=8 (4 Yes, 4 No) | 0.5000 | 0.4583 |
| 31.0 | n=5 (4 Yes, 1 No) | 0.3200 | n=7 (3 Yes, 4 No) | 0.4898 | 0.4190 |
| 33.5 | n=6 (5 Yes, 1 No) | 0.2778 | n=6 (2 Yes, 4 No) | 0.4444 | 0.3611 |
| **36.5** | **n=7 (6 Yes, 1 No)** | **0.2449** | **n=5 (1 Yes, 4 No)** | **0.3200** | **0.2762** |
| 39.0 | n=8 (6 Yes, 2 No) | 0.3750 | n=4 (1 Yes, 3 No) | 0.3750 | 0.3750 |
| 42.5 | n=9 (6 Yes, 3 No) | 0.4444 | n=3 (1 Yes, 2 No) | 0.4444 | 0.4444 |
| 46.5 | n=10 (6 Yes, 4 No) | 0.4800 | n=2 (1 Yes, 1 No) | 0.5000 | 0.4833 |
| 49.0 | n=11 (7 Yes, 4 No) | 0.4628 | n=1 (0 Yes, 1 No) | 0.0000 | 0.4242 |

The best threshold is 36.5.

Weighted Gini:

$$\frac{7}{12}\left(\frac{12}{49}\right)+\frac{5}{12}\left(\frac{8}{25}\right)=\frac{29}{105}\approx 0.2762$$

This age split ties the Income split (same weighted Gini of 0.2762), but it is not as good as Marital Status. It does not perfectly separate the labels. Even at its best threshold ($36.5$), Age ($0.2762$) still underperforms Marital Status ($0.1389$). Therefore, Marital Status remains the best root split.

---
# Problem 2: K-Means Clustering

Points: P1 (2, 3), P2 (5, 8), P3 (1, 2), P4 (7, 6), P5 (4, 9), P6 (3, 5)

## a. Pairwise Euclidean distances

$$d=\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$$

#### **Squared Distances (the value under the square root) $(x_2-x_1)^2 + (y_2-y_1)^2$** 

| | P1 | P2 | P3 | P4 | P5 | P6 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **P1** | 0 | 34 | 2 | 34 | 40 | 5 |
| **P2** | 34 | 0 | 52 | 8 | 2 | 13 |
| **P3** | 2 | 52 | 0 | 52 | 58 | 13 |
| **P4** | 34 | 8 | 52 | 0 | 18 | 17 |
| **P5** | 40 | 2 | 58 | 18 | 0 | 17 |
| **P6** | 5 | 13 | 13 | 17 | 17 | 0 |

#### **Euclidean Pairwise Distances $\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}$**

| | P1 | P2 | P3 | P4 | P5 | P6 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| **P1** | 0.000 | 5.831 | 1.414 | 5.831 | 6.325 | 2.236 |
| **P2** | 5.831 | 0.000 | 7.211 | 2.828 | 1.414 | 3.606 |
| **P3** | 1.414 | 7.211 | 0.000 | 7.211 | 7.616 | 3.606 |
| **P4** | 5.831 | 2.828 | 7.211 | 0.000 | 4.243 | 4.123 |
| **P5** | 6.325 | 1.414 | 7.616 | 4.243 | 0.000 | 4.123 |
| **P6** | 2.236 | 3.606 | 3.606 | 4.123 | 4.123 | 0.000 |

## b. K-Means with K = 2

Initialize centroids at P1 = (2, 3) and P2 = (5, 8).

### Initialized with $C_1 = P1(2, 3)$ and $C_2 = P2(5, 8)$.

#### **Iteration 1**

| Point | Coordinates | Distance to $C_1(2, 3)$ | Distance to $C_2(5, 8)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **0.000** | $5.831$ | **1** |
| **P2** | $(5, 8)$ | $5.831$ | **0.000** | **2** |
| **P3** | $(1, 2)$ | **1.414** | $7.211$ | **1** |
| **P4** | $(7, 6)$ | $5.831$ | **2.828** | **2** |
| **P5** | $(4, 9)$ | $6.325$ | **1.414** | **2** |
| **P6** | $(3, 5)$ | **2.236** | $3.606$ | **1** |

**Centroid Update:**
- **New $C_1^{(1)}$** = mean of $\{P_1, P_3, P_6\} = \left(\frac{2+1+3}{3}, \frac{3+2+5}{3}\right) = (2, 3.333)$
- **New $C_2^{(1)}$** = mean of $\{P_2, P_4, P_5\} = \left(\frac{5+7+4}{3}, \frac{8+6+9}{3}\right) = (5.333, 7.667)$

---

#### **Iteration 2**

| Point | Coordinates | Distance to $C_1(2.0, 3.333)$ | Distance to $C_2(5.333, 7.667)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **0.333** | $5.735$ | **1** |
| **P2** | $(5, 8)$ | $4.784$ | **0.471** | **2** |
| **P3** | $(1, 2)$ | **1.667** | $7.134$ | **1** |
| **P4** | $(7, 6)$ | $5.667$ | **2.357** | **2** |
| **P5** | $(4, 9)$ | $6.009$ | **1.886** | **2** |
| **P6** | $(3, 5)$ | **1.944** | $3.543$ | **1** |

Since no points changed clusters in iteration 2, the centroids remain the same and the algorithm has converged.

**Converged result ($K=2$):**
- Cluster 1: $\{P_1,P_3,P_6\}$ with centroid $(2,3.333)$
- Cluster 2: $\{P_2,P_4,P_5\}$ with centroid $(5.333,7.667)$

## c. K = 3

### Case 1: initialize c1 = P1 (2, 3), c2 = P2 (5, 8), c3 = P3 (1, 2)

#### **Iteration 1**

| Point | Coordinates | Distance to $C_1(2, 3)$ | Distance to $C_2(5, 8)$ | Distance to $C_3(1, 2)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **0.000** | $5.831$ | $1.414$ | **1** |
| **P2** | $(5, 8)$ | $5.831$ | **0.000** | $7.211$ | **2** |
| **P3** | $(1, 2)$ | $1.414$ | $7.211$ | **0.000** | **3** |
| **P4** | $(7, 6)$ | $5.831$ | **2.828** | $7.211$ | **2** |
| **P5** | $(4, 9)$ | $6.325$ | **1.414** | $7.616$ | **2** |
| **P6** | $(3, 5)$ | **2.236** | $3.606$ | $3.606$ | **1** |

**Centroid Update:**
- **New $C_1^{(1)}$** = mean of $\{P_1, P_6\} = (2.5, 4.0)$
- **New $C_2^{(1)}$** = mean of $\{P_2, P_4, P_5\} = (5.333, 7.667)$
- **New $C_3^{(1)}$** = mean of $\{P_3\} = (1.0, 2.0)$

---

#### **Iteration 2**

| Point | Coordinates | Distance to $C_1(2.5, 4.0)$ | Distance to $C_2(5.333, 7.667)$ | Distance to $C_3(1.0, 2.0)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **1.118** | $5.735$ | $1.414$ | **1** |
| **P2** | $(5, 8)$ | $4.717$ | **0.471** | $7.211$ | **2** |
| **P3** | $(1, 2)$ | $2.500$ | $7.134$ | **0.000** | **3** |
| **P4** | $(7, 6)$ | $4.924$ | **2.357** | $7.211$ | **2** |
| **P5** | $(4, 9)$ | $5.220$ | **1.886** | $7.616$ | **2** |
| **P6** | $(3, 5)$ | **1.118** | $3.543$ | $3.606$ | **1** |


In iteration 2, P1 is 1.118 from c1 and 1.414 from c3, so it stays in cluster 1. P6 is 1.118 from c1, so it stays too. P2, P4 and P5 remain closest to c2. Since all points remain in their respective clusters, the algorithm has **converged**..

**Final result:**
- Cluster 1: $\{P_1,P_6\}$ with centroid $(2.5,4.0)$
- Cluster 2: $\{P_2,P_4,P_5\}$ with centroid $(5.333,7.667)$
- Cluster 3: $\{P_3\}$ with centroid $(1.0,2.0)$

### Case 2: initialize c1 = P1 (2, 3), c2 = P2 (5, 8), c3 = P5 (4, 9)

#### **Iteration 1**

| Point | Coordinates | Distance to $C_1(2, 3)$ | Distance to $C_2(5, 8)$ | Distance to $C_3(4, 9)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **0.000** | $5.831$ | $6.325$ | **1** |
| **P2** | $(5, 8)$ | $5.831$ | **0.000** | $1.414$ | **2** |
| **P3** | $(1, 2)$ | **1.414** | $7.211$ | $7.616$ | **1** |
| **P4** | $(7, 6)$ | $5.831$ | **2.828** | $4.243$ | **2** |
| **P5** | $(4, 9)$ | $6.325$ | $1.414$ | **0.000** | **3** |
| **P6** | $(3, 5)$ | **2.236** | $3.606$ | $4.123$ | **1** |

**Centroid Update:**
- **New $C_1^{(1)}$** = mean of $\{P_1, P_3, P_6\} = (2.0, 3.333)$
- **New $C_2^{(1)}$** = mean of $\{P_2, P_4\} = (6.0, 7.0)$
- **New $C_3^{(1)}$** = mean of $\{P_5\} = (4.0, 9.0)$

---

#### **Iteration 2**

| Point | Coordinates | Distance to $C_1(2.0, 3.333)$ | Distance to $C_2(6.0, 7.0)$ | Distance to $C_3(4.0, 9.0)$ | Cluster Assignment |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **P1** | $(2, 3)$ | **0.333** | $5.657$ | $6.325$ | **1** |
| **P2** | $(5, 8)$ | $5.548$ | **1.414** | **1.414** | **2** (tie rule) |
| **P3** | $(1, 2)$ | **1.667** | $7.071$ | $7.616$ | **1** |
| **P4** | $(7, 6)$ | $5.667$ | **1.414** | $4.243$ | **2** |
| **P5** | $(4, 9)$ | $6.009$ | $2.828$ | **0.000** | **3** |
| **P6** | $(3, 5)$ | **1.944** | $3.606$ | $4.123$ | **1** |

In iteration 2, P1, P3 and P6 stay in cluster 1. P4 stays in cluster 2 (1.414 vs 4.243). P2 is exactly 1.414 from both c2 and c3, which is a tie; here it keeps its current cluster (cluster 2).

Since all points remain in their respective clusters, the algorithm has **converged**.

**Final result:**
- Cluster 1: $\{P_1,P_3,P_6\}$ with centroid $(2.0,3.333)$
- Cluster 2: $\{P_2,P_4\}$ with centroid $(6.0,7.0)$
- Cluster 3: $\{P_5\}$ with centroid $(4.0,9.0)$

### Findings: Choosing the initial step

K-Means is sensitive to where initial centroids are placed. In both cases, selecting two initial centroids very close to one another ($d(P_1,P_3)=1.414$ in Case 1; $d(P_2,P_5)=1.414$ in Case 2) splits a natural group in half and leaves an isolated singleton cluster ($\{P_3\}$ in Case 1, $\{P_5\}$ in Case 2), rather than separating the three visually distinct regions of the dataset: bottom-left $\{P_1,P_3,P_6\}$, top $\{P_2,P_5\}$, and right $\{P_4\}$.

---
# Problem 3: Neural Network on Problem 1 Data

## a. Variance of Age and Income

**Formulas**

- Mean: $\bar{x}=\frac{1}{n}\sum_{i=1}^{n} x_i$
- Population variance: $\sigma^2=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2$
- Sample variance: $s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2$
- $n=12$

In [4]:
age    = data["Age"].to_numpy()
income = data["Income"].to_numpy()
n = len(age)

def variance_report(name, x):
    x = [int(v) for v in x]
    mean = Fraction(sum(x), n)
    ssd  = sum((Fraction(v) - mean) ** 2 for v in x)      # sum of squared deviations
    s2, sigma2 = ssd / (n - 1), ssd / n
    print(f"{name}")
    print(f"  Values: {x}")
    print(f"  Mean = {sum(x)}/{n} = {mean} ≈ {float(mean):.4f}")
    print(f"  Sum of squared deviations = {ssd} ≈ {float(ssd):.4f}")
    print(f"  Sample variance     s^2     = ({ssd})/{n-1} = {s2} ≈ {float(s2):.4f}")
    print(f"  Population variance sigma^2 = ({ssd})/{n} = {sigma2} ≈ {float(sigma2):.4f}")
    print()
    return float(s2), float(sigma2)

s2_age, p2_age = variance_report("Variance of Age", age)
s2_inc, p2_inc = variance_report("Variance of Income ($1000)", income)

# cross-check with numpy
print("numpy check, Age:    ", np.var(age, ddof=1).round(4), np.var(age).round(4))
print("numpy check, Income: ", np.var(income, ddof=1).round(4), np.var(income).round(4))
print(f"\nIncome variance / Age variance = {s2_inc / s2_age:.2f}")

Variance of Age
  Values: [25, 45, 30, 35, 50, 28, 40, 32, 38, 22, 29, 48]
  Mean = 422/12 = 211/6 ≈ 35.1667
  Sum of squared deviations = 2747/3 ≈ 915.6667
  Sample variance     s^2     = (2747/3)/11 = 2747/33 ≈ 83.2424
  Population variance sigma^2 = (2747/3)/12 = 2747/36 ≈ 76.3056

Variance of Income ($1000)
  Values: [50, 100, 60, 40, 90, 70, 120, 55, 80, 65, 55, 95]
  Mean = 880/12 = 220/3 ≈ 73.3333
  Sum of squared deviations = 18800/3 ≈ 6266.6667
  Sample variance     s^2     = (18800/3)/11 = 18800/33 ≈ 569.6970
  Population variance sigma^2 = (18800/3)/12 = 4700/9 ≈ 522.2222

numpy check, Age:     83.2424 76.3056
numpy check, Income:  569.697 522.2222

Income variance / Age variance = 6.84


**Results:**
- Age: sample variance $s^2_{\text{Age}}=\frac{2747}{33}\approx 83.2424$, population variance $\sigma^2_{\text{Age}}=\frac{2747}{36}\approx 76.3056$
- Income: sample variance $s^2_{\text{Income}}=\frac{18800}{33}\approx 569.6970$, population variance $\sigma^2_{\text{Income}}=\frac{4700}{9}\approx 522.2222$

Income has a much larger variance than Age (about 7 times larger).

## b. Network architecture

Encoding: Single = 0, Married = 1; Buy: Yes = 1, No = 0. All $f_i$ are sigmoid functions.

- **Input layer:** 3 feature nodes (age, income, and Marital) plus 1 bias node (constant value +1).
- **Hidden layer:** 2 hidden neurons (labeled with pre-activations $z_1, z_2$ and activations $f_1, f_2$) plus 1 bias node (constant value $+1$).
- **Output layer:** 1 output neuron (labeled with pre-activation $z_3$ and activation $f_3$).

## c. Deriving $\hat{y}$

With $f(z)=\frac{1}{1+e^{-z}}$:

$$z_1=w_1+w_3\cdot\text{age}+w_5\cdot\text{income}+w_7\cdot\text{marital},\qquad f_1=f(z_1)$$

$$z_2=w_2+w_4\cdot\text{age}+w_6\cdot\text{income}+w_8\cdot\text{marital},\qquad f_2=f(z_2)$$

$$z_3=w_9+w_{10}\,f_1+w_{11}\,f_2$$

$$\hat{y}=f_3(z_3)=\frac{1}{1+e^{-z_3}}$$

$\hat{y}$ is the predicted probability that the customer buys. Predict Yes (1) if $\hat{y}>0.5$, otherwise No (0).

## d. Forward propagation (first 5 customers)

Using the weights $w_1$ to $w_{11}$: $[1, 2, 1, -2, 3, 4, 1, -3, 1, 2, -2]$

| Customer | Age | Income | Marital | $z_1$ | $z_2$ | $f_1$ | $f_2$ | $z_3$ | $\hat{y}$ | Pred | $y$ (Buy) |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | $25$ | $50$ | $0$ (Single) | $176$ | $152$ | $1.000$ | $1.000$ | $1.000$ | $0.731$ | **1** | **1** (Yes) |
| **2** | $45$ | $100$ | $1$ (Married) | $347$ | $309$ | $1.000$ | $1.000$ | $1.000$ | $0.731$ | **1** | **0** (No) |
| **3** | $30$ | $60$ | $0$ (Single) | $211$ | $182$ | $1.000$ | $1.000$ | $1.000$ | $0.731$ | **1** | **1** (Yes) |
| **4** | $35$ | $40$ | $1$ (Married) | $157$ | $89$ | $1.000$ | $1.000$ | $1.000$ | $0.731$ | **1** | **1** (Yes) |
| **5** | $50$ | $90$ | $1$ (Married) | $322$ | $259$ | $1.000$ | $1.000$ | $1.000$ | $0.731$ | **1** | **0** (No) |

## e. Accuracy on the 5 examples

Predictions (from part d) are 1, 1, 1, 1, 1 against true labels 1, 0, 1, 1, 0. Customers 1, 3 and 4 are correct; customers 2 and 5 are wrong.

$$\text{Accuracy}=\frac{\text{correct}}{\text{total}}=\frac{3}{5}=60\%$$

## f. Cross-entropy vs. MSE

1. **No vanishing gradient at the output.** With MSE, $\frac{\partial L}{\partial z_3}=(\hat{y}-y)\,\hat{y}(1-\hat{y})$. The factor $\hat{y}(1-\hat{y})$ goes to 0 when the sigmoid saturates, so a confidently wrong prediction (e.g. $\hat{y}\approx 0$ when $y=1$) learns very slowly. With cross-entropy, the sigmoid derivative cancels:
$$\frac{\partial L}{\partial z_3}=\hat{y}-y$$
so the gradient is largest exactly when the error is largest.
2. **Heavier penalty for confident mistakes:** $\log(\hat{y})\to\infty$ as $\hat{y}\to 0$ for $y=1$, while MSE is capped at $\frac{1}{2}$.
3. **Probabilistic meaning.** Cross-entropy is the negative log-likelihood of a Bernoulli (binary) label, so minimizing it is maximum-likelihood estimation. It also gives a convex loss for logistic regression, whereas MSE with a sigmoid is not convex.

### g. MSE single sample
For **Customer 1** ($\text{age} = 25,\ \text{income} = 50,\ \text{marital} = 0,\ y = 1$) using MSE loss $L = \frac{1}{2}(y - \hat{y})^2$ and sigmoid activations $\sigma'(z) = \sigma(z)(1 - \sigma(z))$:

#### 1. Chain Rule Derivation
Tracing the dependency path $w_3 \to z_1 \to f_1 \to z_3 \to \hat{y} \to L$:

$$\frac{\partial L}{\partial w_3} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_3} \cdot \frac{\partial z_3}{\partial f_1} \cdot \frac{\partial f_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_3}$$

Computing each partial derivative:
1. $\dfrac{\partial L}{\partial \hat{y}} = \dfrac{\partial}{\partial \hat{y}}\left[\frac{1}{2}(y - \hat{y})^2\right] = -(y - \hat{y}) = \hat{y} - y$
2. $\dfrac{\partial \hat{y}}{\partial z_3} = f_3'(z_3) = \hat{y}(1 - \hat{y})$
3. $\dfrac{\partial z_3}{\partial f_1} = \dfrac{\partial}{\partial f_1}\left(w_9 + w_{10}f_1 + w_{11}f_2\right) = w_{10}$
4. $\dfrac{\partial f_1}{\partial z_1} = f_1'(z_1) = f_1(1 - f_1)$
5. $\dfrac{\partial z_1}{\partial w_3} = \dfrac{\partial}{\partial w_3}\left(w_1 + w_3\cdot\text{age} + w_5\cdot\text{income} + w_7\cdot\text{marital}\right) = \text{age}$

Combining all five factors:

$$\frac{\partial L}{\partial w_3} = (\hat{y} - y) \cdot \hat{y}(1 - \hat{y}) \cdot w_{10} \cdot f_1(1 - f_1) \cdot \text{age}$$

#### 2. Numerical Evaluation (Customer 1)
From part (d) for Customer 1: $z_1 = 176$, $f_1 = \frac{1}{1 + e^{-176}} \approx 1$, $z_3 = 1$, $\hat{y} = \frac{1}{1 + e^{-1}} \approx 0.7311$, $y = 1$, $w_{10} = 2$, and $\text{age} = 25$:

* $\dfrac{\partial L}{\partial \hat{y}} = 0.7311 - 1 = -0.2689$
* $\dfrac{\partial \hat{y}}{\partial z_3} = 0.7311(1 - 0.7311) = 0.1966$
* $\dfrac{\partial z_3}{\partial f_1} = w_{10} = 2$
* $\dfrac{\partial f_1}{\partial z_1} = f_1(1 - f_1) = \dfrac{e^{-176}}{(1 + e^{-176})^2} \approx 3.69 \times 10^{-77} \approx \mathbf{0}$
* $\dfrac{\partial z_1}{\partial w_3} = 25$

$$\frac{\partial L}{\partial w_3} = (-0.2689)(0.1966)(2)\left(3.69 \times 10^{-77}\right)(25) \approx -4.88 \times 10^{-76} $$ (effectively zero)

### h. All Gradients $\frac{\partial L}{\partial w_i}$ ($i = 1 \dots 11$)

#### Calculated Gradients for Customer 1
Using $\delta_3 = (\hat{y} - y)\hat{y}(1 - \hat{y}) \approx -0.0529$:

| Layer | Weight | Gradient Formula | Value |
| :--- | :---: | :--- | :---: |
| **Input $\to$ Hidden 1** | $w_1$ (bias) | $\delta_3 \cdot w_{10} \cdot f_1(1 - f_1) \cdot 1$ | $\approx \mathbf{0}$ ($-1.95 \times 10^{-77}$) |
| | $w_3$ (age) | $\delta_3 \cdot w_{10} \cdot f_1(1 - f_1) \cdot 25$ | $\approx \mathbf{0}$ ($-4.88 \times 10^{-76}$) |
| | $w_5$ (income) | $\delta_3 \cdot w_{10} \cdot f_1(1 - f_1) \cdot 50$ | $\approx \mathbf{0}$ ($-9.76 \times 10^{-76}$) |
| | $w_7$ (marital) | $\delta_3 \cdot w_{10} \cdot f_1(1 - f_1) \cdot 0$ | $\mathbf{0}$ |
| **Input $\to$ Hidden 2** | $w_2$ (bias) | $\delta_3 \cdot w_{11} \cdot f_2(1 - f_2) \cdot 1$ | $\approx \mathbf{0}$ ($+1.03 \times 10^{-66}$) |
| | $w_4$ (age) | $\delta_3 \cdot w_{11} \cdot f_2(1 - f_2) \cdot 25$ | $\approx \mathbf{0}$ ($+2.58 \times 10^{-65}$) |
| | $w_6$ (income) | $\delta_3 \cdot w_{11} \cdot f_2(1 - f_2) \cdot 50$ | $\approx \mathbf{0}$ ($+5.17 \times 10^{-65}$) |
| | $w_8$ (marital) | $\delta_3 \cdot w_{11} \cdot f_2(1 - f_2) \cdot 0$ | $\mathbf{0}$ |
| **Hidden $\to$ Output** | $w_9$ (bias) | $\delta_3 \cdot 1$ | **$-0.0529$** |
| | $w_{10}$ ($f_1$) | $\delta_3 \cdot f_1$ | **$-0.0529$** |
| | $w_{11}$ ($f_2$) | $\delta_3 \cdot f_2$ | **$-0.0529$** |

#### Observations & Analysis
1. **Vanishing Gradients in the First Layer ($w_1$ through $w_8$):**
   * All gradients feeding into the hidden layer ($w_1 \dots w_8$) are **effectively $0$** ($\sim 10^{-65}$ to $10^{-77}$), meaning the hidden-layer weights will not update at all during SGD.
   * **Root Cause (Sigmoid Saturation from Unscaled Inputs):** Because `Age` ($25$) and `Income` ($50$) are not normalized and are multiplied by large initial weights ($w_3=1, w_5=3, w_6=4$), the pre-activations $z_1 = 176$ and $z_2 = 152$ fall far into the flat right tail of the sigmoid function ($f_1 \approx 1, f_2 \approx 1$). At saturation, the sigmoid derivative $f_i(1 - f_i) \approx 0$, blocking any gradient from flowing backward into the first layer. (Additionally, $w_7$ and $w_8$ equal exact $0$ because $\text{marital} = 0$ for Customer 1.)
2. **Non-Zero Output Layer Gradients ($w_9, w_{10}, w_{11}$):**
   * Only the output-layer weights ($w_9, w_{10}, w_{11}$) receive a meaningful non-zero gradient ($-0.0529$). Because $z_3 = 1$ lies in the active, non-saturated region of the output sigmoid ($\hat{y} \approx 0.7311$), $\hat{y}(1 - \hat{y}) \approx 0.1966$ remains sizable, and since $f_1 = f_2 = 1$, all three output weights share the exact same gradient $\delta_3 \approx -0.0529$.
3. **Practical Remedy:**
   * Standardizing/normalizing `Age` and `Income` (e.g., to mean $0$, variance $1$) and initializing weights with smaller random values (such as Xavier/Glorot initialization) keeps $z_1$ and $z_2$ near $0$, preventing sigmoid saturation and allowing gradients to propagate into the hidden layer.